# Combine class masks

Purpose:
--------
The purpose of this notebook is to combine class masks generated for multiple years by CGNet in order to output a heat map of AR frequency

Authors/Contributors:
---------------------
* Teagan King

## Import libraries

In [3]:
import numpy as np
import glob
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import netCDF4 as nc
import matplotlib.image as mpimg

In [1]:
def visualize_frequency_map_scaled(frequency_map, title, colorbar_text, filepath, background_image):
    """Save a PNG of frequency_map with title and colorbar_text at filepath"""

    # Square figure
    plt.figure(figsize=(15, 12), dpi=300)
    plt.rc('xtick', labelsize=20)
    plt.rc('ytick', labelsize=20)
    plt.title(title, fontdict={'fontsize': 24})

    # Background image stretched
    plt.imshow(
        plt.imread(background_image),
        extent=[-180, 180, -90, 90],
        aspect='auto'   # stretch vertically to fill square
    )

    # Grid for lon/lat
    lon = np.linspace(-180, 180, frequency_map.shape[1])
    lat = np.linspace(-90, 90, frequency_map.shape[0])
    lon, lat = np.meshgrid(lon, lat)

    # Mask frequencies
    masked_freq_map = np.ma.masked_array(frequency_map, mask=(frequency_map == 0)).astype(float)

    # Contours
    contourf = plt.contourf(
        lon, lat, masked_freq_map,
        levels=np.linspace(frequency_map.min(), frequency_map.max(), 11),
        alpha=0.7,
        cmap='RdYlBu'
    )

    # Colorbar
    cbar = plt.colorbar(contourf, orientation='vertical',
                        ticks=np.linspace(0, frequency_map.max(), 3))
    cbar.ax.set_ylabel(colorbar_text, size=20)

    # Remove equal-aspect restriction → let it stretch!
    # plt.gca().set_aspect('equal', 'box')
    plt.xlim(-180, 180)
    plt.ylim(-90, 90)

    plt.savefig(filepath, bbox_inches="tight", facecolor='w')
    plt.close()

## Antarctic Heat Map:

In [3]:
var_name = "masks"
for channel_name in ['psl_tmq']:
    # loop through ensembles for SH:
    for ensemble in ['BRCP26C5CN', 'BRCP85C5CN', 'B20TRC5CN']:
        combined_mask = None
        # loop through various years
        year_number = 0
        for f in glob.glob(f'/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/{ensemble}/*/masks_{channel_name}/*'):
            year_number+=1
            with nc.Dataset(f, "r") as ds:
                data = ds.variables[var_name][:]
                masked_data = np.ma.masked_invalid(data)

                # Collapse time dimension -> sum over axis 0 to include all masks from all timestamps in a given year
                summed = masked_data.sum(axis=0)   # shape (1152, 1152)

                # Accumulate into combined_mask
                if combined_mask is None:
                    combined_mask = summed
                else:
                    combined_mask += summed
                    print(f'added mask for {f}')
        combined_mask = combined_mask/(year_number*12)
        print('running vis freq map')
        visualize_frequency_map_scaled(combined_mask,
                                       f"Frequency map of ARs from {ensemble}",
                                       "Frequency of events per month",
                                       f"/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/{ensemble}/combined_class_masks_{channel_name}_month.png",
                                       '/glade/work/tking/cgnet/ClimateNet/climatenet/bluemarble_fake/BM_Polar_SPS.jpeg')

        print(f'made {ensemble} {channel_name} Antarctic combined class mask')

added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2014/masks_psl_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2015/masks_psl_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2010/masks_psl_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2011/masks_psl_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2012/masks_psl_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2013/masks_psl_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2007/masks_psl_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tk

## Arctic Heat Map:
#### Note: kernel will die upon opening ds.variables[var_name][:] with default resources-- request more memory! maybe also take out '[:]'?

In [11]:
# loop through ensembles for NH:
channel_name = 'tmq'
var_name = "masks"
for channel_name in ['tmq', 'psl']:
    for ensemble in ['BRCP85C5CN', 'BRCP26C5CN', 'B20TRC5CN']:
        combined_mask = None
        # loop through various years
        for f in glob.glob(f'/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/{ensemble}/*/masks/*'):
            with nc.Dataset(f, "r") as ds:
                data = ds.variables[var_name][:]
                masked_data = np.ma.masked_invalid(data)
                # Collapse time dimension -> sum over axis 0 to include all masks from all timestamps in a given year
                summed = masked_data.sum(axis=0)   # shape (1152, 1152)
    
                # Accumulate into combined_mask
                if combined_mask is None:
                    combined_mask = summed
                else:
                    combined_mask += summed
                    print(f'added mask for {f}')
    
        combined_mask = combined_mask/(year_number*12)
        print('running vis freq map')
        visualize_frequency_map_scaled(combined_mask,
                                       f"Frequency map of ARs from {ensemble}",
                                       "Number of events",
                                       f"/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/{ensemble}/combined_masks_{ensemble}/combined_class_masks_{channel_name}_month.png",
                                       '/glade/work/tking/cgnet/ClimateNet/climatenet/bluemarble/GL180.jpeg')
        print(f'made {ensemble} Arctic combined class mask')

added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP85C5CN/2090/masks/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP85C5CN/2091/masks/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP85C5CN/2092/masks/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP85C5CN/2093/masks/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP85C5CN/2088/masks/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP85C5CN/2089/masks/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP85C5CN/2098/masks/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP8

# Generate Antarctic Difference Plots:

In [ ]:
var_name = "masks"
# loop through ensembles for SH:
for ensemble in ['BRCP85C5CN', 'B20TRC5CN', 'BRCP26C5CN']:
    combined_mask = None
    # loop through various years
    year_number = 0
    for f in glob.glob(f'/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/{ensemble}/*/masks_tmq/*'):
        year_number+=1
        with nc.Dataset(f, "r") as ds:
            data = ds.variables[var_name][:]
            masked_data = np.ma.masked_invalid(data)

            # Collapse time dimension -> sum over axis 0 to include all masks from all timestamps in a given year
            summed = masked_data.sum(axis=0)   # shape (1152, 1152)

            # Accumulate into combined_mask
            if combined_mask is None:
                combined_mask = summed
            else:
                combined_mask += summed
                print(f'added mask for {f}')
    combined_mask_tmq = combined_mask/(year_number*12)
    year_number = 0
    combined_mask = None
    for f in glob.glob(f'/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/{ensemble}/*/masks_psl_tmq/*'):
        year_number+=1
        with nc.Dataset(f, "r") as ds:
            data = ds.variables[var_name][:]
            masked_data = np.ma.masked_invalid(data)

            # Collapse time dimension -> sum over axis 0 to include all masks from all timestamps in a given year
            summed = masked_data.sum(axis=0)   # shape (1152, 1152)

            # Accumulate into combined_mask
            if combined_mask is None:
                combined_mask = summed
            else:
                combined_mask += summed
                print(f'added mask for {f}')
    combined_mask_psl_tmq = combined_mask/(year_number*12)
    combined_mask_tmq_minus_psl_tmq = combined_mask_tmq - combined_mask_psl_tmq

    print('running vis freq map')
    visualize_frequency_map_scaled(combined_mask_tmq_minus_psl_tmq,
                                f"Frequency map of ARs from {ensemble}",
                                "Frequency of events per month",
                                f"/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/{ensemble}/combined_class_masks_tmq_minus_psl_tmq_month.png",
                                '/glade/work/tking/cgnet/ClimateNet/climatenet/bluemarble_fake/BM_Polar_SPS.jpeg')

    print(f'made {ensemble} tmq_minus_psl_tmq Antarctic combined class mask')

added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2090/masks_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2091/masks_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2092/masks_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2093/masks_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2088/masks_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2089/masks_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2098/masks_tmq/class_masks.nc
added mask for /glade/campaign/cgd/ccr/tking/cgnet/model_inference_da